# Hospital Dataset Cleaning

Clean and validate the messy hospital patient-visit dataset.

In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)

input_path = "Hospital_Messy_20.csv"

df = pd.read_csv(input_path)
print("Loaded shape:", df.shape)

Loaded shape: (20, 9)


## Standardize Text and Missing Values

In [2]:
str_cols = [
    col for col in df.columns
    if pd.api.types.is_string_dtype(df[col])
]
for col in str_cols:
    df[col] = df[col].astype("string").str.strip()

missing_tokens = {"", "na", "n/a", "null", "none", "unknown", "not available", "-"}
for col in str_cols:
    df[col] = df[col].mask(df[col].str.lower().isin(missing_tokens))

print("Missing values after standardization:")
print(df.isna().sum())

Missing values after standardization:
PatientID      0
PatientName    1
Gender         0
Age            0
SystolicBP     1
DiastolicBP    4
HeartRate      2
VisitDate      0
City           1
dtype: int64


## Normalize Patient Categories

In [ ]:
df["Gender"] = (
    df["Gender"]
    .str.upper()
    .map({"F": "Female", "FEMALE": "Female", "M": "Male", "MALE": "Male"})
)

print("Gender values:")
print(df["Gender"].value_counts(dropna=False))

Gender values:
Gender
Female    12
Male       8
Name: count, dtype: int64
Cities: ['Bangalore', 'Delhi', 'Hyderabad', 'Mumbai', 'Pune', 'Unknown']


In [ ]:
df["PatientName"] = df["PatientName"].fillna("Unknown")

print("Missing patient names:", int(df["PatientName"].isna().sum()))

In [ ]:
df["City"] = df["City"].fillna("Unknown")

print("Cities:", sorted(df["City"].dropna().unique()))

## Clean Ages, Blood Pressure, and Heart Rates

In [ ]:
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df.loc[~df["Age"].between(0, 120), "Age"] = np.nan
df["Age"] = df["Age"].fillna(df["Age"].median()).round().astype(int)

print(df["Age"].describe())

             Age  SystolicBP  DiastolicBP  HeartRate
count  20.000000   20.000000     20.00000  20.000000
mean   44.100000  121.000000     77.10000  82.500000
std    14.182643    8.091581      7.36921   9.127748
min    20.000000  106.000000     65.00000  65.000000
25%    30.750000  114.500000     73.00000  77.000000
50%    48.000000  122.000000     76.00000  84.000000
75%    54.000000  126.500000     83.25000  91.000000
max    67.000000  134.000000     94.00000  95.000000


In [ ]:
raw_systolic = df["SystolicBP"].astype("string")
bp_parts = raw_systolic.str.extract(r"^(\d+(?:\.\d+)?)\s*/\s*(\d+(?:\.\d+)?)$")
combined_bp = bp_parts[0].notna()
df["SystolicBP"] = pd.to_numeric(raw_systolic.where(~combined_bp, bp_parts[0]), errors="coerce")
df.loc[~df["SystolicBP"].between(70, 220), "SystolicBP"] = np.nan
df["SystolicBP"] = df["SystolicBP"].fillna(df["SystolicBP"].median()).round().astype(int)

print(df["SystolicBP"].describe())

In [ ]:
df["DiastolicBP"] = pd.to_numeric(df["DiastolicBP"].where(~combined_bp, bp_parts[1]), errors="coerce")
df.loc[~df["DiastolicBP"].between(40, 130), "DiastolicBP"] = np.nan
df["DiastolicBP"] = df["DiastolicBP"].fillna(df["DiastolicBP"].median()).round().astype(int)

print(df["DiastolicBP"].describe())

In [ ]:
df["HeartRate"] = pd.to_numeric(df["HeartRate"], errors="coerce")
df.loc[~df["HeartRate"].between(40, 220), "HeartRate"] = np.nan
df["HeartRate"] = df["HeartRate"].fillna(df["HeartRate"].median()).round().astype(int)

print(df["HeartRate"].describe())

## Normalize Visit Dates

In [5]:
df["VisitDate"] = pd.to_datetime(
    df["VisitDate"],
    format="mixed",
    errors="coerce"
)

print("Invalid or missing dates:", df["VisitDate"].isna().sum())
df["VisitDate"] = df["VisitDate"].dt.strftime("%Y-%m-%d")

Invalid or missing dates: 0


## Remove Duplicate Patients and Validate

In [6]:
duplicate_count = df["PatientID"].duplicated().sum()
df = df.drop_duplicates(subset="PatientID", keep="first").reset_index(drop=True)

assert df["PatientID"].is_unique
assert df["Gender"].dropna().isin(["Female", "Male"]).all()
assert df["Age"].between(0, 120).all()
assert df["SystolicBP"].between(70, 220).all()
assert df["DiastolicBP"].between(40, 130).all()
assert df["HeartRate"].between(40, 220).all()
assert df["VisitDate"].notna().all()

print("Duplicates removed:", duplicate_count)
print("Missing values:")
print(df.isna().sum())

Duplicates removed: 0
Missing values:
PatientID      0
PatientName    0
Gender         0
Age            0
SystolicBP     0
DiastolicBP    0
HeartRate      0
VisitDate      0
City           0
dtype: int64


## Save the Cleaned Dataset

In [ ]:
output_path = "Hospital_Cleaned_20.csv"
df.to_csv(output_path, index=False)

cleaned_df = pd.read_csv(output_path)
print("Saved to:", output_path)
print("Saved shape:", cleaned_df.shape)

Saved to: c:\Users\Dharshan\Desktop\Data_Cleaning\Hospital\Hospital_Cleaned_20.csv
Saved shape: (20, 9)
